In [1]:
# ============================================================
# MODEL TRAINING — Leakage-Filtered Features + Multi-Metric Eval (2 Models)
# ============================================================

import os, json
from datetime import datetime
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# ------------------------------------------------------------
# 1️⃣ Initialize Spark
# ------------------------------------------------------------
spark = (
    SparkSession.builder
    .appName("LoanDefault_LeakageFiltered")
    .getOrCreate()
)

# ------------------------------------------------------------
# 2️⃣ Load all Gold feature store files
# ------------------------------------------------------------
GOLD_DIR = "/app/datamart/gold/feature_store/gold_feature_store_*.parquet"
MODEL_BANK_DIR = "/app/model_bank"
os.makedirs(MODEL_BANK_DIR, exist_ok=True)

gold_df = (
    spark.read
    .option("mergeSchema", "true")
    .parquet(GOLD_DIR)
)

print("✅ Loaded Gold data:", gold_df.count(), "rows")

# ------------------------------------------------------------
# 3️⃣ Add file_date column from file name (for OOT)
# ------------------------------------------------------------
gold_df = gold_df.withColumn("_input_file", F.input_file_name())
gold_df = gold_df.withColumn(
    "file_date",
    F.to_date(
        F.regexp_extract(F.col("_input_file"), r"gold_feature_store_(\d{4}_\d{2}_\d{2})", 1),
        "yyyy_MM_dd"
    )
).drop("_input_file")

print("✅ Distinct file dates:")
gold_df.select("file_date").distinct().orderBy("file_date").show(30, False)

# ------------------------------------------------------------
# 4️⃣ Label hygiene
# ------------------------------------------------------------
gold_df = gold_df.withColumn("label", F.col("label").cast(DoubleType()))
gold_df = gold_df.filter(F.col("label").isin(0.0, 1.0))

# ------------------------------------------------------------
# 5️⃣ Define OOT split (Oct–Dec 2024)
# ------------------------------------------------------------
OOT_START, OOT_END = "2024-10-01", "2024-12-31"
oot_df = gold_df.filter((F.col("file_date") >= OOT_START) & (F.col("file_date") <= OOT_END))
dev_df = gold_df.filter(F.col("file_date") < OOT_START)

print("Rows → OOT:", oot_df.count(), "| Dev:", dev_df.count())

# ------------------------------------------------------------
# 6️⃣ Customer-level split
# ------------------------------------------------------------
unique_customers = (
    gold_df.select("Customer_ID").distinct().withColumn("rand", F.rand(seed=42))
)
train_cut, val_cut = 0.70, 0.85
train_ids = unique_customers.filter(F.col("rand") <= train_cut).select("Customer_ID")
val_ids   = unique_customers.filter((F.col("rand") > train_cut) & (F.col("rand") <= val_cut)).select("Customer_ID")
test_ids  = unique_customers.filter(F.col("rand") > val_cut).select("Customer_ID")

train_df = dev_df.join(train_ids, "Customer_ID", "inner")
val_df   = dev_df.join(val_ids,   "Customer_ID", "inner")
test_df  = dev_df.join(test_ids,  "Customer_ID", "inner")

print("Split sizes → Train:", train_df.count(),
      "| Val:", val_df.count(),
      "| Test:", test_df.count(),
      "| OOT:", oot_df.count())

# ------------------------------------------------------------
# 7️⃣ Feature filtering — remove leaky features
# ------------------------------------------------------------
numeric_types = {"double", "float", "integer", "long", "short", "decimal"}
exclude_cols = {
    "Customer_ID","loan_id","snapshot_date","application_date",
    "gold_processing_timestamp","feature_store_version","dpd_threshold",
    "label","file_date"
}
leakage_patterns = ["dpd", "overdue", "balance", "ever", "mob", "delin", "arrear"]

feature_cols = [
    f.name for f in gold_df.schema.fields
    if (f.name not in exclude_cols)
    and (f.dataType.typeName() in numeric_types)
    and all(p not in f.name.lower() for p in leakage_patterns)
]

print("Feature count after leakage filter:", len(feature_cols))
print("Sample of features:", feature_cols[:10])

# ------------------------------------------------------------
# 8️⃣ Handle missing/invalids
# ------------------------------------------------------------
fill_map = {c: 0 for c in feature_cols}
for df_name, df_var in [("train", train_df), ("val", val_df), ("test", test_df), ("oot", oot_df)]:
    locals()[f"{df_name}_df"] = df_var.fillna(fill_map)
    for c in feature_cols:
        locals()[f"{df_name}_df"] = locals()[f"{df_name}_df"].withColumn(
            c, F.when(F.isnan(F.col(c)) | F.col(c).isNull(), 0).otherwise(F.col(c))
        )

# ------------------------------------------------------------
# 9️⃣ Train and compare multiple models (LogReg vs RandomForest)
# ------------------------------------------------------------
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw", handleInvalid="keep")
scaler    = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

models_to_train = {
    "LogisticRegression": LogisticRegression(
        featuresCol="features", labelCol="label",
        maxIter=50, regParam=0.1, elasticNetParam=0.0
    ),
    "RandomForest": RandomForestClassifier(
        featuresCol="features", labelCol="label",
        numTrees=100, maxDepth=8, seed=42
    )
}

metrics_all = {}

for model_name, estimator in models_to_train.items():
    print(f"\n🚀 Training {model_name} ...")
    pipeline = Pipeline(stages=[assembler, scaler, estimator])
    model = pipeline.fit(train_df)

    evaluator_auc  = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
    evaluator_acc  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    evaluator_f1   = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
    evaluator_prec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
    evaluator_rec  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

    metrics = {}
    for name, df in [("Validation", val_df), ("Test", test_df), ("OOT", oot_df)]:
        pred = model.transform(df)
        auc = evaluator_auc.evaluate(pred)
        acc = evaluator_acc.evaluate(pred)
        f1  = evaluator_f1.evaluate(pred)
        prec = evaluator_prec.evaluate(pred)
        rec = evaluator_rec.evaluate(pred)
        metrics[name] = {
            "AUC": round(auc, 4),
            "Accuracy": round(acc, 4),
            "F1": round(f1, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4)
        }
        print(f"{name} → AUC={auc:.4f} | Acc={acc:.4f} | F1={f1:.4f} | Prec={prec:.4f} | Rec={rec:.4f}")

    # Save model
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_path = os.path.join(MODEL_BANK_DIR, f"{model_name.lower()}_clean_{timestamp}")
    model.write().overwrite().save(model_path)

    metrics_all[model_name] = metrics

# ------------------------------------------------------------
# 🔟 Save metrics to metadata file
# ------------------------------------------------------------
meta_file = os.path.join(MODEL_BANK_DIR, f"metrics_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
with open(meta_file, "w") as f:
    json.dump(metrics_all, f, indent=2)

print("\n✅ Models and metrics saved in:", MODEL_BANK_DIR)
print(f"   ├─ Individual models in subfolders")
print(f"   └─ Summary metrics file: {meta_file}")

# ------------------------------------------------------------
# 11️⃣ Summary comparison printout
# ------------------------------------------------------------
print("\n📊 Model Performance Summary")
for mname, mvals in metrics_all.items():
    print(f"\n=== {mname} ===")
    for split, vals in mvals.items():
        print(f"{split}: AUC={vals['AUC']} | Acc={vals['Accuracy']} | F1={vals['F1']} | Prec={vals['Precision']} | Rec={vals['Recall']}")

# ------------------------------------------------------------
# 12️⃣ File-date range per subset
# ------------------------------------------------------------
print("\nFile-date range per subset:")
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df), ("OOT", oot_df)]:
    df.select(
        F.min("file_date").alias("min_file_date"),
        F.max("file_date").alias("max_file_date")
    ).show()

# Note: keep Spark alive for diagnostics
# spark.stop()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/09 05:55:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

✅ Loaded Gold data: 11974 rows
✅ Distinct file dates:


25/11/09 05:55:35 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

+----------+
|file_date |
+----------+
|2023-01-01|
|2023-02-01|
|2023-03-01|
|2023-04-01|
|2023-05-01|
|2023-06-01|
|2023-07-01|
|2023-08-01|
|2023-09-01|
|2023-10-01|
|2023-11-01|
|2023-12-01|
|2024-01-01|
|2024-02-01|
|2024-03-01|
|2024-04-01|
|2024-05-01|
|2024-06-01|
|2024-07-01|
|2024-08-01|
|2024-09-01|
|2024-10-01|
|2024-11-01|
|2024-12-01|
+----------+



Rows → OOT: 1459 | Dev: 10515


...

+-------------+-------------+
|min_file_date|max_file_date|
+-------------+-------------+
|   2023-01-01|   2024-09-01|
+-------------+-------------+



+-------------+-------------+
|min_file_date|max_file_date|
+-------------+-------------+
|   2023-01-01|   2024-09-01|
+-------------+-------------+



+-------------+-------------+
|min_file_date|max_file_date|
+-------------+-------------+
|   2023-01-01|   2024-09-01|
+-------------+-------------+



[Stage 343:>                                                        (0 + 8) / 8]

+-------------+-------------+
|min_file_date|max_file_date|
+-------------+-------------+
|   2024-10-01|   2024-12-01|
+-------------+-------------+

